# Décodage — imports

Charge le modèle et les fonctions de décodage.

In [1]:
import sys
from pathlib import Path

import torch

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.llm.model import load_model
from src.llm.decoding import (
    apply_temperature,
    apply_top_k,
    apply_top_p,
    sample_next_token,
)

# Chargement du modèle

Récupère le tokenizer, le modèle et le device.

In [2]:
llm = load_model()

tokenizer = llm.tokenizer
model = llm.model
device = llm.device

Chargement du modèle : Qwen/Qwen2.5-3B-Instruct
Device utilisé : cuda


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

# Logits du prochain token

Calcule les scores produits après le prompt.

In [3]:
text = "Paris est la capitale de"

inputs = tokenizer(
    text,
    return_tensors="pt",
).to(device)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits[0, -1, :]

logits.shape

torch.Size([151936])

# Température

Observe son effet sur la distribution des probabilités.

In [4]:
temperatures = [0.2, 0.7, 1.0, 1.5, 2.0]

for temperature in temperatures:

    scaled_logits = apply_temperature(
        logits,
        temperature,
    )

    probabilities = torch.softmax(
        scaled_logits,
        dim=-1,
    )

    top_probs, top_ids = torch.topk(
        probabilities,
        k=5,
    )

    print(f"\nTemperature = {temperature}")

    for prob, token_id in zip(
        top_probs.tolist(),
        top_ids.tolist(),
    ):
        print(
            repr(tokenizer.decode([token_id])),
            round(prob, 4),
        )


Temperature = 0.2
' la' 1.0
' l' 0.0
' France' 0.0
' quel' 0.0
' (' 0.0

Temperature = 0.7
' la' 0.9297
' l' 0.0522
' France' 0.0132
' quel' 0.0007
' (' 0.0007

Temperature = 1.0
' la' 0.7812
' l' 0.1055
' France' 0.0388
' quel' 0.0052
' (' 0.0052

Temperature = 1.5
' la' 0.3418
' l' 0.0918
' France' 0.0461
' quel' 0.0125
' (' 0.0125

Temperature = 2.0
' la' 0.0742
' l' 0.0273
' France' 0.0166
' quel' 0.0061
' (' 0.0061


# Top-k

Ne conserve que les `k` tokens les mieux classés.

In [5]:
filtered_logits = apply_top_k(
    logits,
    k=5,
)

probabilities = torch.softmax(
    filtered_logits,
    dim=-1,
)

non_zero = torch.where(
    probabilities > 0
)[0]

print("Nombre de tokens possibles :", len(non_zero))

for token_id in non_zero:
    print(
        repr(tokenizer.decode([token_id.item()])),
        probabilities[token_id].item(),
    )

Nombre de tokens possibles : 5
' (' 0.005615234375
' l' 0.11279296875
' la' 0.8359375
' France' 0.04150390625
' quel' 0.005615234375


# Top-p

Conserve les tokens jusqu’au seuil de probabilité cumulée.

In [6]:
filtered_logits = apply_top_p(
    logits,
    p=0.9,
)

probabilities = torch.softmax(
    filtered_logits,
    dim=-1,
)

remaining = torch.where(
    probabilities > 0
)[0]

print(
    "Nombre de tokens conservés :",
    len(remaining),
)

Nombre de tokens conservés : 3


# Sampling

Échantillonne plusieurs tokens selon la distribution filtrée.

In [7]:
for _ in range(10):

    token_id = sample_next_token(
        logits,
        temperature=0.8,
        top_k=50,
        top_p=0.9,
    )

    print(
        repr(
            tokenizer.decode(
                token_id.tolist()
            )
        )
    )

' la'
' l'
' la'
' la'
' l'
' la'
' la'
' la'
' la'
' la'


# Génération complète

Compare plusieurs températures sur une réponse complète.

In [8]:
from src.llm.decoding import generate


prompt = "Explique simplement ce qu'est un Transformer."
print("température faible")
print(
    generate(
        model,
        tokenizer,
        prompt,
        device,
        temperature=0.2,
        top_k=50,
        top_p=0.9,
    )
)

print("température moyenne")
print(
    generate(
        model,
        tokenizer,
        prompt,
        device,
        temperature=0.8,
        top_k=50,
        top_p=0.9,
    )
)

print("température élevée")
print(
    generate(
        model,
        tokenizer,
        prompt,
        device,
        temperature=1.5,
        top_k=50,
        top_p=0.9,
    )
)

température faible
Explique simplement ce qu'est un Transformer. Un Transformer est un type de réseau de neurones artificiels utilisé principalement pour le traitement de séquences, comme les textes ou les séquences de mots. Il est composé de couches de couches de transfert, qui
température moyenne
Explique simplement ce qu'est un Transformer. Un Transformer est un type d'unité de traitement de l'information pour les langues naturelles. C'est un modèle de langage qui comprend une série de couches de réseaux de neurones, où chaque couche app
température élevée
Explique simplement ce qu'est un Transformer. Un Transformer est un type de modèle de prédiction supervisée très populaires, généralement utilisé en apprentissage automatique. C'est souvent comparable à une machine de langue naturelle. Il est utilisé pour déterminer l'abs
